In [5]:
print("Soluciones encontradas:")
for x1 in range(1, 3):
  for x2 in range(1, 3):
    if x1 != x2:
      for x3 in range(1, 4):
        if x1 != x3 and x2 != x3:
          if 3*x1 + x2 + x3 == 10:
            print(f"x1 = {x1}, x2 = {x2}, x3 = {x3}")

Soluciones encontradas:
x1 = 2, x2 = 1, x3 = 3


## 2 Latin Square

### 1. Solve it by Brute Force

In [5]:
def generar_permutaciones(lista):
  # Genera todas las permutaciones de una lista (recursivo)
  if len(lista) == 0:
    return [[]]
  resultado = []
  
  for i in range(len(lista)):
    elem = lista[i]
    resto = lista[:i] + lista[i+1:]

    for p in generar_permutaciones(resto):
      resultado.append([elem] + p)
  return resultado

def es_latino(cuadro):
  n = len(cuadro)
  # Verificar columnas
  for col in range(n):
    vistos = set()
    
    for fila in range(n):
      if cuadro[fila][col] in vistos:
        return False
      
      vistos.add(cuadro[fila][col])
  return True

def latin_square(n):
  # Generamos todas las permutaciones posibles en una fila
  filas_posibles = generar_permutaciones(list(range(1, n + 1)))

  soluciones = []
  # Generamos todas las combinaciones de filas
  def backtrack(cuadro):
    if len(cuadro) == n:
      if es_latino(cuadro):
        soluciones.append([fila[:] for fila in cuadro])
      return
    for fila in filas_posibles:
      cuadro.append(fila)
      if es_latino(cuadro):  # poda temprana
        backtrack(cuadro)
      cuadro.pop()
  
  backtrack([])
  return soluciones

In [ ]:
# Latin Square de 4x4 usando BF
n = 4
soluciones = latin_square(n)

print(f"Latin Squares de {n}x{n} encontrados: {len(soluciones)}")

for s in soluciones[:5]:
  for fila in s:
    print(fila)
  print()

Latin Squares de 4x4 encontrados: 576
[1, 2, 3, 4]
[2, 1, 4, 3]
[3, 4, 1, 2]
[4, 3, 2, 1]

[1, 2, 3, 4]
[2, 1, 4, 3]
[3, 4, 2, 1]
[4, 3, 1, 2]

[1, 2, 3, 4]
[2, 1, 4, 3]
[4, 3, 1, 2]
[3, 4, 2, 1]

[1, 2, 3, 4]
[2, 1, 4, 3]
[4, 3, 2, 1]
[3, 4, 1, 2]

[1, 2, 3, 4]
[2, 3, 4, 1]
[3, 4, 1, 2]
[4, 1, 2, 3]



### 2. Solve it by constraint programming

In [ ]:
#!pip install ortools

You should consider upgrading via the 'C:\Python310\python.exe -m pip install --upgrade pip' command.


  Using cached ortools-9.15.6755-cp310-cp310-win_amd64.whl (23.8 MB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
  Using cached immutabledict-4.3.1-py3-none-any.whl (5.0 kB)
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl (437 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl (135 kB)


You should consider upgrading via the 'C:\Python310\python.exe -m pip install --upgrade pip' command.


In [1]:
from ortools.sat.python import cp_model

def latin_square_cp(n):
  # Creamos el modelo
  model = cp_model.CpModel()

  # Variables: matriz NxN con valores entre 1 y n
  cuadro = [[model.NewIntVar(1, n, f"x_{i}_{j}") for j in range(n)] for i in range(n)]

  # Restricciones: cada fila debe tener todos los valores distintos
  for i in range(n):
    model.AddAllDifferent(cuadro[i])

  # Restricciones: cada columna debe tener todos los valores distintos
  for j in range(n):
    col = [cuadro[i][j] for i in range(n)]
    model.AddAllDifferent(col)
  
  # Creamos el solver
  solver = cp_model.CpSolver()

  # Callback para imprimir soluciones
  class LatinSquarePrinter(cp_model.CpSolverSolutionCallback):
    def __init__(self, cuadro, n, limite=5):
      cp_model.CpSolverSolutionCallback.__init__(self)
      self.cuadro = cuadro
      self.n = n
      self.contador = 0
      self.limite = limite

    def OnSolutionCallback(self):
      if self.contador < self.limite:
        print(f"Solución {self.contador+1}:")
        for i in range(self.n):
          fila = [self.Value(self.cuadro[i][j]) for j in range(self.n)]
          print(fila)
        print()
      self.contador += 1
  
  # Buscamos soluciones
  printer = LatinSquarePrinter(cuadro, n, limite=5)
  solver.SearchForAllSolutions(model, printer)

  print(f"Se encontraron {printer.contador} soluciones en total.")

In [2]:
# Latin Square de 4x4 usando CP
latin_square_cp(4)

Solución 1:
[3, 4, 1, 2]
[2, 1, 4, 3]
[4, 3, 2, 1]
[1, 2, 3, 4]

Solución 2:
[3, 4, 1, 2]
[2, 3, 4, 1]
[4, 1, 2, 3]
[1, 2, 3, 4]

Solución 3:
[4, 3, 1, 2]
[2, 1, 4, 3]
[3, 4, 2, 1]
[1, 2, 3, 4]

Solución 4:
[3, 4, 1, 2]
[4, 1, 2, 3]
[2, 3, 4, 1]
[1, 2, 3, 4]

Solución 5:
[3, 4, 1, 2]
[4, 3, 2, 1]
[2, 1, 4, 3]
[1, 2, 3, 4]

Se encontraron 576 soluciones en total.
